# 04 — Wall Detection and the Liquidity Map

**Strategy reference:** §9.6 (Wall detection), §9.7 (Evidence →
state), §10 (Liquidity Map).

Walls are the resting-liquidity concentrations Ripple trades
*against* (bounce) or *through* (breakout).  The detector is
deterministic: it scans the top N levels per side, computes the
median depth, and flags every level whose depth exceeds
`median × wall_depth_multiple` (default ×3).

In [ ]:
# ── Data-source configuration ─────────────────────────────────────────
# OHLCV (Parquet) — S3 or local, controlled by DATA_STORE env var:
#   Local (default):  reads <project_root>/data/ohlcv/...
#   S3:               uncomment the two lines below
# import os
# os.environ["DATA_STORE"] = "s3"
# os.environ["S3_BUCKET"]  = "trading-data-centheos"
#
# Tick data (HDF5) — always stored locally; pull from S3 on demand:
#   load_ticks(...)              → use local cache (fast, no network)
#   load_ticks(..., refresh=True) → sync from S3 then read (ETag-gated)
#   Requires: AWS_PROFILE=trading (or AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY)
import os
os.environ["AWS_PROFILE"] = "trading"
os.environ["S3_BUCKET"]   = "trading-data-centheos"
# ─────────────────────────────────────────────────────────────────────

import sys, importlib
from pathlib import Path

_here = Path.cwd().resolve()
for _cand in [_here, *_here.parents]:
    if (_cand / "schemas.py").exists():
        _root = _cand; break
else:
    raise RuntimeError("Could not locate project root (no schemas.py found)")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import notebooks.utils as _utils_mod
importlib.reload(_utils_mod)   # always pick up on-disk changes without restarting the kernel

from notebooks.utils import (
    load_ohlcv, list_ohlcv, load_ticks, latest_book,
    plot_ohlcv, plot_equity_curve, configure_pandas, env_summary,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

configure_pandas()
%matplotlib inline

In [ ]:
from schemas import RippleConfig, WallSide
cfg = RippleConfig()
cfg

## 1. Load order book snapshots

In [ ]:
ticks = # refresh=False (default) — use local cache, no network access.
# refresh=True            — ETag-check S3 and download only if collector
#                           has uploaded new data since last refresh.
load_ticks('BTCUSDT', max_depth_snapshots=20_000, max_trades=2_000, max_depth_updates=2_000,
                   refresh=False)
snaps = ticks['depth_snapshots']
print('depth_snapshots rows:', len(snaps))

## 2. Wall detection on a single snapshot (§9.6.1)
```
median_depth = median(depths_in_top_N_levels)
threshold    = median_depth × wall_depth_multiple
wall_levels  = [lvl for lvl in levels if lvl.qty >= threshold]
```

In [ ]:
def detect_walls_simple(book_side: pd.DataFrame, multiple: float = 3.0):
    """Return the subset of levels classified as walls."""
    if book_side.empty:
        return book_side
    median_depth = float(book_side['quantity'].median())
    threshold = median_depth * multiple
    walls = book_side[book_side['quantity'] >= threshold].copy()
    walls['threshold'] = threshold
    walls['median_depth'] = median_depth
    walls['ratio'] = walls['quantity'] / median_depth
    return walls

bids, asks, ts_ms = latest_book(snaps, max_levels=cfg.wall_scan_depth)
bid_walls = detect_walls_simple(bids, cfg.wall_depth_multiple)
ask_walls = detect_walls_simple(asks, cfg.wall_depth_multiple)
print(f'bid walls: {len(bid_walls)},  ask walls: {len(ask_walls)}')
pd.concat({'bid_walls': bid_walls, 'ask_walls': ask_walls})

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(bids['price'], bids['quantity'], height=1.0, color='#1f9d55', alpha=0.5, label='bid depth')
ax.barh(asks['price'], -asks['quantity'], height=1.0, color='#cc1f1a', alpha=0.5, label='ask depth')
if not bid_walls.empty:
    ax.barh(bid_walls['price'], bid_walls['quantity'], height=1.0,
            edgecolor='#0a4d28', linewidth=2, fill=False, label='bid wall')
if not ask_walls.empty:
    ax.barh(ask_walls['price'], -ask_walls['quantity'], height=1.0,
            edgecolor='#7a0e0a', linewidth=2, fill=False, label='ask wall')
ax.set_title(f'Wall detection — snapshot @ ts={ts_ms}')
ax.set_xlabel('quantity (ask negative)'); ax.set_ylabel('price')
ax.legend(loc='upper right'); ax.grid(alpha=0.3); plt.show()

## 3. Wall quality score $W_p$ (§9.5.7)
$$W_p = w_{\mathcal D}\cdot\frac{\mathcal D_p}{\text{median}(\mathcal D)}
       + w_\pi \cdot \pi_p + w_c \cdot (1 - c_p)
       + w_{\text{prox}}\cdot \frac{1}{1+|p-P_t|/\hat\sigma_P}$$

We sweep each component to build intuition for the score.

In [ ]:
def wall_quality(depth_ratio, persistence, cancel_rate, distance_sigma, weights=(0.25,0.25,0.25,0.25)):
    w = weights
    return (w[0]*depth_ratio + w[1]*persistence + w[2]*(1-cancel_rate)
            + w[3] * 1.0 / (1.0 + distance_sigma))

rng = np.linspace(0, 5, 50)
fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
axes[0].plot(rng, [wall_quality(d, 0.5, 0.2, 0.5) for d in rng]); axes[0].set_title('vs depth_ratio')
axes[1].plot(np.linspace(0,1,50), [wall_quality(2.0, p, 0.2, 0.5) for p in np.linspace(0,1,50)]); axes[1].set_title('vs persistence')
axes[2].plot(np.linspace(0,1,50), [wall_quality(2.0, 0.5, c, 0.5) for c in np.linspace(0,1,50)]); axes[2].set_title('vs cancel_rate')
axes[3].plot(np.linspace(0,5,50), [wall_quality(2.0, 0.5, 0.2, x) for x in np.linspace(0,5,50)]); axes[3].set_title('vs distance (σ)')
for ax in axes: ax.axhline(cfg.wall_min_quality, ls='--', color='#888'); ax.grid(alpha=0.3); ax.set_ylabel('W_p')
fig.suptitle('Wall quality sensitivity (dashed = wall_min_quality default 0.5)', y=1.05)
plt.tight_layout(); plt.show()

## 4. Track a single wall over time
Pick a high-depth bid level and follow its depth + cancellation
across all snapshots in the window.  Cancellation proxy = drop in
depth that wasn't matched by trades at that price.

In [ ]:
if bid_walls is None or bid_walls.empty:
    print('no bid walls found in this snapshot — try a wider window')
else:
    target_price = float(bid_walls.iloc[0]['price'])
    series = (snaps[(snaps['side'] == 0) & (np.isclose(snaps['price'], target_price))]
                .sort_values('timestamp')
                .set_index('timestamp')['quantity'])
    print(f'tracking bid level p={target_price}, observations={len(series)}')
    if len(series) > 1:
        fig, ax = plt.subplots(figsize=(12, 3))
        ax.plot(pd.to_datetime(series.index, unit='ms'), series.values, color='#1f9d55')
        ax.set_title(f'Depth of bid level @ {target_price}'); ax.set_ylabel('quantity'); ax.grid(alpha=0.3)
        plt.show()

## 5. Evidence scores → state (§9.7)
Four logistic-style scores; the state with the highest score wins
iff it exceeds `state_activation_threshold` AND beats the runner-up
by `state_margin` (else stay in current state — hysteresis).

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def evidence_scores(wall_quality, flow, dp_over_sigma, cvd_slope, depth_delta_ratio,
                    trade_vol_at_wall, persistence):
    return {
        'ABSORPTION': sigmoid(1.0*wall_quality + 0.6*flow - 0.8*abs(dp_over_sigma)),
        'EXHAUSTION': sigmoid(-1.0*cvd_slope*np.sign(flow) + 0.6*(1.0 - flow)),
        'WITHDRAWAL': sigmoid(2.0*depth_delta_ratio - 1.0*trade_vol_at_wall),
        'REFILL':     sigmoid(2.0*depth_delta_ratio + 0.8*persistence),
    }

def infer_state(scores, current='STABLE',
                activation=cfg.state_activation_threshold, margin=cfg.state_margin):
    items = sorted(scores.items(), key=lambda kv: -kv[1])
    best_name, best = items[0]
    if best < activation:
        return 'STABLE'
    second = items[1][1]
    if best - second < margin:
        return current
    return best_name

# Walkthrough examples
for scenario in [
    ('wall absorbs flow',  dict(wall_quality=1.5, flow=0.9, dp_over_sigma=0.1, cvd_slope=0.0, depth_delta_ratio=0.0, trade_vol_at_wall=0.5, persistence=0.9)),
    ('exhausting trend',   dict(wall_quality=0.2, flow=0.6, dp_over_sigma=1.0, cvd_slope=-0.05, depth_delta_ratio=0.0, trade_vol_at_wall=0.1, persistence=0.3)),
    ('wall pulled',        dict(wall_quality=0.0, flow=0.2, dp_over_sigma=2.0, cvd_slope=0.0, depth_delta_ratio=0.8, trade_vol_at_wall=0.0, persistence=0.0)),
    ('depth refilling',    dict(wall_quality=0.0, flow=0.0, dp_over_sigma=0.0, cvd_slope=0.0, depth_delta_ratio=0.6, trade_vol_at_wall=0.0, persistence=0.8)),
]:
    name, kw = scenario
    s = evidence_scores(**kw)
    print(f'{name:20s} → {infer_state(s)}    scores={ {k: round(v,2) for k,v in s.items()} }')

## 6. Liquidity Map snapshot (§10)
Bring the pieces together: walls, void corridors, the VWAP
anchor, plus a destination score per level.

In [ ]:
def liquidity_map(bids, asks, mid):
    bid_walls = detect_walls_simple(bids, cfg.wall_depth_multiple)
    ask_walls = detect_walls_simple(asks, cfg.wall_depth_multiple)
    lm = pd.concat([
        bid_walls.assign(side='BID'),
        ask_walls.assign(side='ASK'),
    ])
    if lm.empty:
        return lm
    sigma_P = max(asks['price'].std() if len(asks) > 1 else 1.0, 1.0)
    lm['dist_sigma'] = (lm['price'] - mid).abs() / sigma_P
    lm['quality']    = lm.apply(
        lambda r: wall_quality(r['ratio'], 0.6, 0.2, r['dist_sigma']), axis=1
    )
    lm['hold_score'] = (lm['quality'] - 0.5).clip(0, 1)
    lm['dest_score'] = (lm['quality'] / (1 + lm['dist_sigma'])).clip(0, 1)
    return lm

if bids.empty or asks.empty:
    print('Skipping: no usable book snapshot in this slice.')
    lm = pd.DataFrame(columns=['price','quality','hold_score','dest_score'])
    mid = float('nan')
else:
    mid = 0.5 * (float(bids['price'].iloc[0]) + float(asks['price'].iloc[0]))
    lm  = liquidity_map(bids, asks, mid)
lm.sort_values('quality', ascending=False) if not lm.empty else lm

In [ ]:
if lm.empty:
    print('Liquidity map empty — skipping plot')
else:
    fig, ax = plt.subplots(figsize=(10, 5))
    for _, row in lm.iterrows():
        color = '#1f9d55' if row['side'] == 'BID' else '#cc1f1a'
        alpha = float(min(1.0, max(0.05, row['quality'])))
        ax.axhline(row['price'], color=color, alpha=alpha, linewidth=2)
    ax.axhline(mid, color='k', linestyle=':', label=f'mid={mid:.2f}')
    ax.set_title('Liquidity map — wall levels coloured by quality')
    ax.set_ylabel('price'); ax.legend(); ax.grid(alpha=0.3); plt.show()

## 7. Void corridors (§10.2.5)
Ranges where both resting depth and traded volume are very low.
Price tends to *travel through* voids quickly.

In [ ]:
full_book = pd.concat([bids.assign(side='BID'), asks.assign(side='ASK')]).sort_values('price')
depth_p10 = full_book['quantity'].quantile(0.1)
voids = full_book[full_book['quantity'] < depth_p10]
print(f'depth-only void candidates (under p10={depth_p10:.2f}): {len(voids)}')
voids.head(10)

## Takeaways

* Wall detection is one-line maths but the *quality score*
  determines whether a wall is tradable.
* Evidence → state has built-in hysteresis (`state_margin`) so the
  output doesn't flicker every tick.
* The liquidity map is what the trade-archetype engine consumes
  to pick targets and define stop-out / scale-out levels.